# Lab 9: Household Energy Consumption Forecasting

**Task:** predict a household's electricity use in the next hour from its recent use.

**Dataset:** Individual Household Electric Power Consumption (UCI: https://archive.ics.uci.edu/dataset/235/individual+household+electric+power+consumption), included as `data/individual_household_electric_power_consumption.zip` as downloaded: one house near Paris, one reading per **minute** from December 2006 to November 2010 (about 2 million rows).

**What we will do:**
1. Parse the date and time, deal with missing readings, and resample the minutes into hours (and look at days).
2. Build supervised features: lag features (previous hours, same hour yesterday) and rolling means (average of the last 24 hours).
3. Train linear regression, Ridge and Lasso, and polynomial regression to show overfitting.
4. Evaluate with $R^2$, look at the error over time, and plot actual against predicted for one week.
5. Explain which lag and rolling features matter most, and how the forecast degrades the further ahead we predict.

The models are the regression models we already know. What's new is how we turn a series of readings into a table of features and targets, and how we test it honestly.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 140)
raw = pd.read_csv("data/individual_household_electric_power_consumption.zip", sep=";", low_memory=False)
print(raw.shape)
raw.head()

`Global_active_power` is the household's total power use in kilowatts (averaged over the minute): our target. The other columns are reactive power, voltage, current intensity and three sub-meters (kitchen, laundry, water heater and air conditioner).

## 1. Dates, missing values and resampling

In [ ]:
print(raw.dtypes)
print(raw["Global_active_power"].head(3).tolist(), "...")
print("rows where the power reading is '?':", (raw["Global_active_power"] == "?").sum())

The numeric columns were read as text, because missing readings are written as `?`. Convert them to numbers (`?` becomes NaN), and combine `Date` (day/month/year, the French format) and `Time` into one timestamp. Giving the format explicitly matters: `01/02/2007` is 1 February here, not 2 January.

In [ ]:
value_cols = raw.columns[2:]
power = raw.copy()
power[value_cols] = power[value_cols].apply(pd.to_numeric, errors="coerce")
power["time"] = pd.to_datetime(power["Date"] + " " + power["Time"], format="%d/%m/%Y %H:%M:%S")
power = power.set_index("time").drop(columns=["Date", "Time"]).sort_index()
print(power.index.min(), "to", power.index.max())
print("missing per column:", power.isna().sum().iloc[:3].to_dict())

In [ ]:
is_na = power["Global_active_power"].isna()
runs = is_na.groupby((is_na != is_na.shift()).cumsum()).sum()[lambda s: s > 0]
print("missing stretches:", len(runs), "| longest:", runs.max(), "minutes =", round(runs.max() / 60 / 24, 1), "days")
print(runs.sort_values(ascending=False).head(5).to_dict())

About 1.25% of the minutes are missing, mostly in a few long outages (the longest is several days). We **resample to hourly averages**, which smooths the noise of individual minutes; an hour with no readings at all stays missing. Short gaps of up to 2 hours are interpolated; longer gaps stay missing and those hours drop out of the table later (inventing days of consumption would add made-up data).

In [ ]:
hourly = power["Global_active_power"].resample("h").mean()
print("hours:", len(hourly), "| hours with no reading:", hourly.isna().sum())
hourly = hourly.interpolate(limit=2, limit_area="inside")
print("still missing after filling gaps up to 2 hours:", hourly.isna().sum())
daily = power["Global_active_power"].resample("D").mean()

fig, axes = plt.subplots(2, 1, figsize=(12, 6))
axes[0].plot(daily, lw=0.8); axes[0].set_title("daily average power (kW)")
week = hourly["2008-01-07":"2008-01-13"]
axes[1].plot(week); axes[1].set_title("one week, hourly (kW)")
plt.tight_layout(); plt.show()

Two patterns are visible: a **yearly** cycle (more electricity in winter: heating and lighting; a dip every August when the family is on holiday) and a **daily** cycle (low at night, peaks in the morning and evening).

## 2. Lag and rolling features

We predict the power in hour $t$ from what is known at the end of hour $t-1$:
- **lag features**: the power 1, 2 and 3 hours before (`lag_1`, `lag_2`, `lag_3`), the same hour yesterday (`lag_24`) and the same hour a week ago (`lag_168`);
- **rolling means**: the average power over the last 24 hours and over the last 7 days (`roll_24`, `roll_168`);
- **calendar features**: hour of the day and day of the week (known in advance for any hour), and the month.

`shift(k)` moves the series down by $k$ rows: row $t$ then holds the value from $k$ hours earlier (our index has exactly one row per hour, so rows are hours).

**The rolling mean must be shifted too.** `hourly.rolling(24).mean()` at row $t$ averages hours $t-23$ to $t$, *including the hour we are trying to predict*. The model would quietly use the answer. Shift first: `hourly.shift(1).rolling(24).mean()` averages hours $t-24$ to $t-1$.

In [ ]:
def make_table(series, horizon=1):
    """Features known `horizon` hours before the target hour."""
    t = pd.DataFrame({"power": series})
    for lag in [horizon, horizon + 1, horizon + 2, 24, 168]:
        if lag >= horizon:
            t[f"lag_{lag}"] = series.shift(lag)
    t["roll_24"] = series.shift(horizon).rolling(24).mean()
    t["roll_168"] = series.shift(horizon).rolling(168).mean()
    t["hour"] = t.index.hour
    t["weekday"] = t.index.dayofweek
    t["month"] = t.index.month
    return t.dropna()

table = make_table(hourly)
print(table.shape)
table.head(3).round(3)

### Splitting in time order
In use, the model forecasts the future from the past, so we test it the same way: train on December 2006 to December 2009, test on 2010. A random split would let the model see the hours just before and after each test hour, which makes the test far too easy.

In [ ]:
split = pd.Timestamp("2010-01-01")
train, test = table[table.index < split], table[table.index >= split]
features = [c for c in table.columns if c != "power"]
print("train:", len(train), "hours | test:", len(test), "hours")

## 3. Models

The calendar features are categories (hour 23 is next to hour 0), so they are one-hot encoded; all numeric features are standardised, which Ridge and Lasso need. Baselines first: a forecast should beat "same as the last hour" (**persistence**) and "same as this hour yesterday".

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import r2_score, mean_absolute_error

calendar = ["hour", "weekday", "month"]
def make_model(features, regressor, degree=1):
    num = [f for f in features if f not in calendar]
    num_steps = [("scale", StandardScaler())] + ([("poly", PolynomialFeatures(degree, include_bias=False))] if degree > 1 else [])
    prep = ColumnTransformer([("num", Pipeline(num_steps), num), ("cal", OneHotEncoder(handle_unknown="ignore"), [f for f in features if f in calendar])])
    return Pipeline([("prep", prep), ("model", regressor)])

results = []
def record(name, p_train, p_test):
    results.append({"model": name, "train R2": r2_score(train["power"], p_train), "test R2": r2_score(test["power"], p_test),
                    "test MAE (kW)": mean_absolute_error(test["power"], p_test)})

record("persistence: lag_1", train["lag_1"], test["lag_1"])
record("same hour yesterday: lag_24", train["lag_24"], test["lag_24"])
linear = make_model(features, LinearRegression()).fit(train[features], train["power"])
record("linear regression", linear.predict(train[features]), linear.predict(test[features]))
cv = TimeSeriesSplit(n_splits=5)                 # tuning folds in time order too
for name, reg in [("ridge", RidgeCV(alphas=np.logspace(-2, 4, 13), cv=cv)), ("lasso", LassoCV(cv=cv, max_iter=20000))]:
    m = make_model(features, reg).fit(train[features], train["power"])
    record(f"{name} (alpha={m[-1].alpha_:.3g})", m.predict(train[features]), m.predict(test[features]))
pd.DataFrame(results).round(3)

Linear regression clearly beats both baselines. Notice that "same hour yesterday" has a *negative* $R^2$: it is worse than simply predicting the average, because the household's routine varies a lot from one day to the next. Ridge and Lasso, with their penalty chosen by time-ordered cross-validation, are no better: 26,000 training hours are plenty for about 50 weights.

Household consumption is hard to predict hour by hour: whether someone switches on the oven or the washing machine in the next hour is close to random, and $R^2$ around 0.5–0.6 is typical for a single home.

### Polynomial regression and overfitting
Squares and products of the numeric features let the model bend and capture interactions (for example, "the last hour matters more in the evening"). Does that help?

In [ ]:
for degree in [2, 3]:
    m = make_model(features, LinearRegression(), degree=degree).fit(train[features], train["power"])
    record(f"polynomial degree {degree}", m.predict(train[features]), m.predict(test[features]))
small = train.iloc[-24 * 60:]                       # the last 60 days of training data only
for degree in [1, 2, 3]:
    m = make_model(features, LinearRegression(), degree=degree).fit(small[features], small["power"])
    results.append({"model": f"degree {degree}, only 60 days of training data", "train R2": r2_score(small["power"], m.predict(small[features])),
                    "test R2": r2_score(test["power"], m.predict(test[features])), "test MAE (kW)": mean_absolute_error(test["power"], m.predict(test[features]))})
pd.DataFrame(results).round(3)

With three years of data, degrees 2 and 3 gain a little on both training and test data: there is enough data for the extra weights, and no sign of overfitting. With only 60 days of data, overfitting is obvious: as the degree rises, the training $R^2$ rises and the test $R^2$ falls sharply. The extra flexibility fits the noise of those particular 60 days.

## 4. Evaluation

In [ ]:
pred = pd.Series(linear.predict(test[features]), index=test.index)
resid = test["power"] - pred
fig, axes = plt.subplots(2, 1, figsize=(12, 7))
wk = slice("2010-02-01", "2010-02-07")
axes[0].plot(test.loc[wk, "power"], label="actual"); axes[0].plot(pred.loc[wk], label="predicted (1 hour ahead)")
axes[0].legend(); axes[0].set_ylabel("kW"); axes[0].set_title("one week of forecasts, February 2010")
monthly_mae = resid.abs().resample("MS").mean()
axes[1].bar(monthly_mae.index.strftime("%b"), monthly_mae.values); axes[1].set_ylabel("mean absolute error (kW)")
axes[1].set_title("error by month of 2010")
plt.tight_layout(); plt.show()
print({d.strftime('%b'): round(v, 3) for d, v in monthly_mae.items()})

- The forecast follows the daily rhythm but **lags behind sudden peaks**: it can only see a spike after it has started. Peaks (cooking, laundry) are exactly the hard part.
- The **error is larger in winter** (more consumption, bigger and more frequent peaks) and smallest in August, when the house is mostly empty. 

## 5. Which features matter, and how far ahead can we forecast?

### Feature importance
Weights of the linear model on standardised features (change in kW per standard deviation, holding the others fixed), and the loss in test $R^2$ when each feature is left out:

In [ ]:
names = [n.split("__")[1] for n in linear[0].get_feature_names_out()]
w = pd.Series(linear[-1].coef_, index=names)
print(w[[f for f in features if f not in calendar]].round(3).sort_values(key=abs, ascending=False))
base = r2_score(test["power"], linear.predict(test[features]))
drops = {}
for f in [f for f in features if f not in calendar] + ["hour"]:
    fs = [x for x in features if x != f]
    m = make_model(fs, LinearRegression()).fit(train[fs], train["power"])
    drops[f] = base - r2_score(test["power"], m.predict(test[fs]))
pd.Series(drops).sort_values(ascending=False).round(4)

**The most recent hour (`lag_1`) matters most by far**: consumption an hour ago is the best single guide to consumption now (an appliance that is on tends to stay on). The other features each add a little once `lag_1` is known; the hour of day and `lag_24` capture the daily routine, and the rolling means the current "level" (cold week, holiday). Because the lags are strongly correlated with each other, leaving one out costs little: the others stand in for it.

### How the forecast degrades further ahead
Forecasting $h$ hours ahead, the most recent value we can use is $h$ hours old. We rebuild the table for each horizon (all lags at least $h$, rolling means ending $h$ hours back) and refit:

In [ ]:
rows = []
for h in [1, 2, 3, 6, 12, 24, 48]:
    t = make_table(hourly, horizon=h)
    tr, te = t[t.index < split], t[t.index >= split]
    f = [c for c in t.columns if c != "power"]
    m = make_model(f, LinearRegression()).fit(tr[f], tr["power"])
    rows.append({"hours ahead": h, "test R2": r2_score(te["power"], m.predict(te[f])), "test MAE (kW)": mean_absolute_error(te["power"], m.predict(te[f]))})
horizon = pd.DataFrame(rows)
plt.plot(horizon["hours ahead"], horizon["test R2"], "o-"); plt.xscale("log"); plt.xlabel("hours ahead"); plt.ylabel("test R2"); plt.show()
horizon.round(3)

The forecast is best one hour ahead and degrades quickly over the first few hours ($R^2$ 0.57 at 1 hour, 0.37 at 3 hours), as the information in the most recent readings goes stale; from about 6 hours on it levels off (about 0.34), because the model then relies on what doesn't go stale: the hour of the day, the day of the week, the season and the same hour on previous days.

**Summary.** Recent consumption (`lag_1`) is the key feature for short-term forecasts; the daily routine (hour, `lag_24`) and the recent level (`roll_24`, `roll_168`) carry what can be predicted further ahead. Any forecasting result should state its horizon, and be tested on a later period than it was trained on.